# NB1 — Knowledge ingestion & vectorization
Scrapes/loads tradition-specific documents, chunks them, embeds with `sentence-transformers`, and stores in a **ChromaDB** vector store on Google Drive.

**REQUIRED- enable Google permissions**

> Run once (or when you add new documents). CPU runtime is fine.

In [1]:
# ── Install dependencies
!pip install -q chromadb sentence-transformers pypdf requests beautifulsoup4 tqdm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 72.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 24.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 91.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.7/18.7 MB 79.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/

In [3]:
# ── Mount Google Drive (persistent storage)
from google.colab import drive
drive.mount('/content/drive')
import os
BASE = '/content/drive/MyDrive/integrative_rl'
os.makedirs(f'{BASE}/chroma_db', exist_ok=True)
os.makedirs(f'{BASE}/raw_docs', exist_ok=True)
os.makedirs(f'{BASE}/episodes', exist_ok=True)
print('Drive mounted. BASE:', BASE)

Mounted at /content/drive
Drive mounted. BASE: /content/drive/MyDrive/integrative_rl


In [4]:
# ── Tradition metadata — extend this dict with your own PDF paths or URLs
TRADITIONS = {
    'TCM': {
        'description': 'Traditional Chinese Medicine — pattern differentiation, acupoints, herbal formulae',
        'sources': [
            # Add PDF paths: f'{BASE}/raw_docs/tcm_materia_medica.pdf'
            # Or public URLs to plain-text summaries
        ],
        'seed_texts': [
            'Liver Qi stagnation presents with wiry pulse, hypochondriac distension, emotional irritability, and purple tongue edges. Treatment principle: soothe liver, move Qi. Formula: Chai Hu Shu Gan San. Key points: LV3, LV14, PC6, GB34.',
            'Kidney Yang deficiency: deep weak pulse especially chi position, cold limbs, frequent pale urination, lower back soreness, fatigue worsening in cold. Formula: Jin Gui Shen Qi Wan. Points: KD3, BL23, CV4, moxa.',
            'Spleen Qi deficiency: soggy pulse, pale swollen tongue with teeth marks, poor appetite, loose stools, fatigue after eating. Formula: Si Jun Zi Tang. Points: SP6, ST36, CV12, BL20.',
            'Phlegm-damp obstruction: slippery pulse, greasy white tongue coat, heaviness, brain fog, nausea, obesity. Formula: Er Chen Tang. Points: ST40, SP9, CV12, PC6.',
            'Heart Blood deficiency: thin pulse, pale tongue, palpitations, insomnia with vivid dreams, poor memory. Formula: Gui Pi Tang. Points: HT7, PC6, BL15, SP6.',
        ]
    },
    'Ayurveda': {
        'description': 'Ayurvedic medicine — dosha theory, rasa, virya, vipaka, panchakarma',
        'sources': [],
        'seed_texts': [
            'Vata excess (Vata Vikruti): dry skin, constipation, variable appetite, anxiety, insomnia, joint cracking, cold sensitivity. Pacify with warm unctuous foods, Ashwagandha (Withania somnifera), Bala, sesame oil Abhyanga, Basti (enema) panchakarma. Avoid cold raw foods, excessive travel, irregular routine.',
            'Pitta excess: yellow greasy tongue coat, sharp rapid pulse, inflammation, skin rashes, acid reflux, irritability, night sweats. Pacify with cooling foods, Shatavari, Amalaki, Brahmi, Neem, Tikta Ghrita. Avoid spicy, fermented, and sour foods. Virechana (purgation) panchakarma.',
            'Kapha excess: thick white tongue coat, slow deep pulse, weight gain, mucus congestion, sluggish digestion, depression, attachment. Use Trikatu, Guggulu, Triphala, dry vigorous exercise, Kapha-reducing fasting. Vamana (emesis) panchakarma when indicated.',
            'Ama accumulation: coated tongue, foul breath, heavy joints, dull appetite, low-grade fatigue. First priority: Deepana-Pachana (kindle digestive fire and digest ama) with Chitrakadi Vati, ginger, long pepper before any tonification.',
            'Meda dhatu imbalance (adipose): Guggulu (Commiphora mukul), Triphala Guggulu, Medohar Guggulu. Reduce Kapha diet. Udvartana (dry powder massage). Assess Agni before supplementing.',
        ]
    },
    'Naturopathy': {
        'description': 'Naturopathic medicine — vis medicatrix naturae, tolle causam, therapeutic order',
        'sources': [],
        'seed_texts': [
            'Naturopathic therapeutic order: (1) Remove obstacles to cure — poor diet, lifestyle, toxic exposures. (2) Stimulate vital force. (3) Support weakened systems. (4) Correct structural integrity. (5) Prescribe specific natural substances. (6) Prescribe pharmacological agents. (7) Surgery.',
            'Hydrotherapy protocol for immune stimulation: Constitutional hydrotherapy — alternating hot and cold towel applications to trunk, 5 min hot / 1 min cold, 3 cycles. Stimulates lymphatic circulation, white blood cell activity, and digestive motility.',
            'Detoxification support: Elimination diet (remove gluten, dairy, soy, corn, eggs, sugar, alcohol for 21 days). Support liver phase I/II with cruciferous vegetables, NAC 600mg BD, milk thistle 140mg TID standardised to 70% silymarin, dandelion root tea.',
            'Stress and adrenal support naturopathic protocol: Adaptogen rotation — Rhodiola rosea (morning), Ashwagandha (evening). Phosphatidylserine 300mg for HPA axis. B-complex, Magnesium glycinate 400mg nocte. Sleep hygiene: dark room, no screens 1hr before, consistent timing.',
            'Gut restoration protocol: Remove (pathogens, irritants), Replace (enzymes, HCl), Reinoculate (probiotics: Lactobacillus rhamnosus GG, Saccharomyces boulardii), Repair (L-glutamine 5g BD, zinc carnosine, slippery elm). 4R protocol foundation.',
        ]
    },
    'Clinical Herbalism': {
        'description': 'Evidence-informed clinical herbalism — Western Eclectic, actions, constituents, interactions',
        'sources': [],
        'seed_texts': [
            'Adaptogens: Withania somnifera (Ashwagandha) — HPA axis modulation, cortisol reduction (Grade A: multiple RCTs). Dose: 300-600mg KSM-66 extract daily. Caution: thyroid conditions, nightshade sensitivity, pregnancy. Interaction: sedatives (additive), thyroid medications.',
            'Nervines: Valeriana officinalis — GABAergic, sleep latency reduction (Grade B). Dose: 300-600mg standardised extract at bedtime or 5ml 1:3 tincture. Caution: hepatotoxicity at high doses long-term. Interaction: benzodiazepines, alcohol (potentiation).',
            'Hepatoprotectives: Silybum marianum (Milk thistle) — silymarin antioxidant, hepatocyte regeneration (Grade A). Dose: 140mg silymarin TID. Safe in pregnancy. Interaction: CYP3A4 substrates (mild inhibition), statins.',
            'Anti-inflammatories: Curcuma longa — NF-kB inhibition, COX-2 inhibition (Grade A). Dose: 500mg curcumin with piperine 5mg TID. Poor bioavailability without lipid or piperine. Interaction: anticoagulants (Warfarin — potentiates), chemotherapy.',
            'Immune modulators: Echinacea purpurea — innate immune stimulation, reduces URI duration (Grade A). Dose: 300mg TID standardised to 4% alkylamides or 5ml 1:2 tincture TID for acute use max 8 weeks. Contraindicated: autoimmune conditions, immunosuppressants.',
            'Cardiovascular: Crataegus monogyna (Hawthorn) — positive inotrope, vasodilator, antioxidant (Grade B). Dose: 160-900mg extract daily. Interaction: digoxin (additive), antihypertensives (additive hypotension). Safe for long-term use.',
        ]
    },
    'Functional Nutrition': {
        'description': 'Functional medicine nutritional protocols — nutrient deficiencies, gut-brain axis, metabolic health',
        'sources': [],
        'seed_texts': [
            'Magnesium deficiency (estimated 50% of Western population): Symptoms — muscle cramps, poor sleep, anxiety, constipation, fatigue, headaches. Testing: RBC magnesium (serum unreliable). Repletion: Magnesium glycinate or malate 200-400mg daily. Food sources: dark leafy greens, pumpkin seeds, dark chocolate.',
            'Vitamin D and immune function: Optimal range 100-150 nmol/L (40-60 ng/mL). Dose to maintain: 2000-4000 IU D3 daily with K2 MK-7 100mcg. Co-factors: magnesium (activates D), zinc. Deficiency linked to autoimmunity, depression, infections.',
            'Omega-3 fatty acids (EPA/DHA): Anti-inflammatory via resolvin and protectin synthesis. Dose: 2-4g combined EPA+DHA daily for anti-inflammatory effect. Fish oil, algae-based DHA for vegans. Monitor if on anticoagulants. Omega-3 index target: >8%.',
            'Gut microbiome and metabolic health: Short-chain fatty acids (butyrate) from fermentable fibre support colonocyte health, reduce intestinal permeability. Target 30+ different plant foods per week. Fermented foods (kefir, kimchi, sauerkraut) increase microbiome diversity (Sonnenburg et al. 2021 Cell).',
            'Mitochondrial support protocol: CoQ10 100-300mg (ubiquinol form preferred over 40yo), B-complex (riboflavin, niacin, thiamine as cofactors), Alpha lipoic acid 300mg, NAC 600mg, magnesium malate. Indicated in fatigue, fibromyalgia, post-viral syndromes.',
            'Blood sugar regulation — functional approach: Chromium picolinate 200-400mcg, Berberine 500mg TID (Grade A: comparable to metformin in T2D, mechanism AMPK activation), Inositol 2-4g (PCOS, insulin resistance), time-restricted eating 16:8, low glycaemic load diet.',
        ]
    }
}
print('Tradition metadata loaded:', list(TRADITIONS.keys()))

Tradition metadata loaded: ['TCM', 'Ayurveda', 'Naturopathy', 'Clinical Herbalism', 'Functional Nutrition']


In [5]:
# ── Embed and store in ChromaDB
import chromadb
from chromadb.utils import embedding_functions
from tqdm import tqdm

emb_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name='all-MiniLM-L6-v2'
)

client = chromadb.PersistentClient(path=f'{BASE}/chroma_db')

for tradition, meta in TRADITIONS.items():
    col_name = tradition.lower().replace(' ', '_').replace('.', '')
    try:
        col = client.get_collection(col_name, embedding_function=emb_fn)
        print(f'Collection {col_name} exists ({col.count()} docs). Skipping.')
    except:
        col = client.create_collection(col_name, embedding_function=emb_fn,
                                        metadata={'tradition': tradition, 'description': meta['description']})

    # Add seed texts
    existing = set(col.get()['ids'])
    to_add_docs, to_add_ids, to_add_meta = [], [], []
    for i, text in enumerate(meta['seed_texts']):
        doc_id = f'seed_{i}'
        if doc_id not in existing:
            to_add_docs.append(text)
            to_add_ids.append(doc_id)
            to_add_meta.append({'tradition': tradition, 'source': 'seed', 'index': i})

    if to_add_docs:
        col.add(documents=to_add_docs, ids=to_add_ids, metadatas=to_add_meta)
        print(f'[{tradition}] Added {len(to_add_docs)} seed texts. Total: {col.count()}')

    # TODO: add PDF ingestion here
    # from pypdf import PdfReader
    # for pdf_path in meta['sources']:
    #     reader = PdfReader(pdf_path)
    #     for pg_num, page in enumerate(reader.pages):
    #         text = page.extract_text()
    #         # chunk into 512-char overlapping windows
    #         chunks = [text[i:i+512] for i in range(0, len(text), 400)]
    #         col.add(documents=chunks, ids=[f'pdf_{pg_num}_{j}' for j in range(len(chunks))],
    #                 metadatas=[{'tradition': tradition, 'source': pdf_path, 'page': pg_num}] * len(chunks))

print('\nAll collections ready:')
for col_name in [t.lower().replace(' ','_').replace('.','') for t in TRADITIONS]:
    col = client.get_collection(col_name, embedding_function=emb_fn)
    print(f'  {col_name}: {col.count()} documents')

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

[TCM] Added 5 seed texts. Total: 5
[Ayurveda] Added 5 seed texts. Total: 5
[Naturopathy] Added 5 seed texts. Total: 5
[Clinical Herbalism] Added 6 seed texts. Total: 6
[Functional Nutrition] Added 6 seed texts. Total: 6

All collections ready:
  tcm: 5 documents
  ayurveda: 5 documents
  naturopathy: 5 documents
  clinical_herbalism: 6 documents
  functional_nutrition: 6 documents


## Done
Collections are persisted in Google Drive at `BASE/chroma_db`. Run **NB2** next.